# Modelagem supervisionada: pré-processamento e baseline (Fase 3)

**Objetivo:** montar o pipeline de pré-processamento e treinar os modelos de referência
(baselines) para prever se um aluno **não** será alfabetizado (classe de risco).

**Entradas:**
- `data/sample/features_aluno`: a camada Gold (1 linha por aluno avaliado em 2024).
- `src/config/features_modelo.json`: as features e o tratamento de cada uma, definidos
  no notebook `02_selecao_features`. Nenhuma lista de features é digitada aqui.

## 1. Setup e carga


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42   # semente fixa: garante que os sorteios se repitam em qualquer execução

CAMINHO_GOLD = "../data/sample/features_aluno"
CAMINHO_DECISOES = Path("../src/config/features_modelo.json")

df = pd.read_parquet(CAMINHO_GOLD)
decisoes = json.loads(CAMINHO_DECISOES.read_text(encoding="utf-8"))

NUMERICAS_LOG = decisoes["numericas_log"]
NUMERICAS_PADRAO = decisoes["numericas_padrao"]
CATEGORICAS = decisoes["categoricas"]
BINARIAS = decisoes["binarias"]
FEATURES = NUMERICAS_LOG + NUMERICAS_PADRAO + CATEGORICAS + BINARIAS

print(f"Gold: {df.shape} | features lidas do JSON: {len(FEATURES)}")

Gold: (1610754, 35) | features lidas do JSON: 24


## 2. Features (X), alvo (y) e grupos

- **X:** as 24 features. As binárias são convertidas de verdadeiro/falso/vazio para
  1/0/vazio, porque o `SimpleImputer` não aceita valores lógicos com vazios
  (seção 8 do notebook 02).
- **y:** o alvo invertido, **1 = não alfabetizado (risco)** (seção 3 do notebook 02).
- **grupos:** o município de cada aluno, usado para que treino e teste nunca
  compartilhem o mesmo município.

In [2]:
X = df[FEATURES].copy()
for coluna in BINARIAS:
    X[coluna] = X[coluna].astype(float)   # True/False/vazio -> 1.0/0.0/NaN

y = (~df[decisoes["alvo"]]).astype(int).rename("nao_alfabetizado")
grupos = df[decisoes["coluna_grupo"]]

print(f"X: {X.shape}")
print(f"Taxa de risco (y = 1): {y.mean():.1%}")
print(f"Municípios: {grupos.nunique()}")

X: (1610754, 24)
Taxa de risco (y = 1): 40.6%
Municípios: 5450


## 3. Divisão entre treino e teste

A divisão é feita **por município**: 20% dos municípios vão para o teste, com todos os
seus alunos. Assim, nenhum município aparece nos dois conjuntos.

**Por que por município:** as features são quase todas municipais e se repetem em todos
os alunos de cada município (pseudo-replicação, `01_eda`, seção 4). Numa divisão
aleatória por aluno, o modelo veria no teste municípios que já conheceu no treino, e a
métrica mediria a capacidade de reconhecer municípios, não de generalizar. A divisão
por município responde à pergunta de negócio: o modelo acerta em municípios que
nunca viu?

**O conjunto de teste é reservado para a avaliação final** (etapa de avaliação do
cronograma). Toda comparação de modelos e ajuste de hiperparâmetros usa validação
cruzada **dentro do treino**.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

divisor = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
idx_treino, idx_teste = next(divisor.split(X, y, groups=grupos))

X_treino, X_teste = X.iloc[idx_treino], X.iloc[idx_teste]
y_treino, y_teste = y.iloc[idx_treino], y.iloc[idx_teste]
grupos_treino, grupos_teste = grupos.iloc[idx_treino], grupos.iloc[idx_teste]

resumo_divisao = pd.DataFrame({
    "alunos": [len(y_treino), len(y_teste)],
    "pct_alunos": [len(y_treino) / len(y), len(y_teste) / len(y)],
    "municipios": [grupos_treino.nunique(), grupos_teste.nunique()],
    "taxa_risco": [y_treino.mean(), y_teste.mean()],
}, index=["treino", "teste"]).round(3)
print(resumo_divisao)

# conferência: nenhum município nos dois conjuntos
assert set(grupos_treino).isdisjoint(set(grupos_teste))

         alunos  pct_alunos  municipios  taxa_risco
treino  1320423        0.82        4360       0.409
teste    290331        0.18        1090       0.393


**Resultado:** o treino tem __ municípios (__% dos alunos) e o teste, __ municípios
(__% dos alunos), sem nenhum município em comum. A taxa de risco é de __% no treino
e __% no teste.

Como 20% dos municípios não correspondem exatamente a 20% dos alunos (os municípios têm
tamanhos muito diferentes), o percentual de alunos no teste varia. A taxa de risco
próxima nos dois conjuntos mostra que o sorteio não desequilibrou o alvo.